# Run Inference — 8 Local Models × 600 Jokes

All-local, all-free pipeline. Eight Ollama-served models, no API keys, no
billing, no rate limits. **Models are stored on the D: drive** (configurable
in Section 2) to avoid filling up the system disk — Ollama's default location
is the user home directory which is usually on C:.

| # | Model | Slug | Paper match | H4 role |
|---|---|---|---|---|
| 1 | DeepSeek-R1-Distill-**Llama**-8B | `r1-distill-llama-8b` | ✅ exact | — |
| 2 | Llama 3.1 8B Instruct | `llama-3.1-8b` | ✅ exact | small Llama |
| 3 | Llama 3.2 3B Instruct | `llama-3.2-3b` | extension | smaller Llama (cross-gen) |
| 4 | Gemma 2 **2B** Instruct | `gemma-2-2b` | extension | small Gemma |
| 5 | Gemma 2 **9B** Instruct | `gemma-2-9b` | extension | large Gemma |
| 6 | Mistral 7B Instruct v0.3 | `mistral-7b` | extension | new family baseline |
| 7 | Phi-3 **Mini** (3.8B) Instruct | `phi-3-mini` | extension | small Phi |
| 8 | Phi-3 **Medium** (14B) Instruct | `phi-3-medium` | extension | large Phi |

**Why this lineup:**
- **Two paper-exact matches** (R1-Distill-Llama-8B, Llama 3.1 8B) — direct comparison to the paper's reported scores
- **Two clean within-family H4 size pairs** — Gemma 2B vs 9B AND Phi-3 Mini vs Medium. Both pairs are same generation, same training run, only size differs. Two independent clean tests of the size hypothesis make the H4 finding much more robust.
- **Cross-generation H4 confound test** (Llama 3.1 8B vs 3.2 3B) — bonus comparison; expected to be noisier than the clean pairs because generation also differs
- **No judge family-bias** — none are Qwen-base, judge has no shared-family inflation risk. DeepSeek-R1's Qwen-distilled variants were deliberately excluded for the same reason.
- **Four model families** — Llama-derived (3 models), Google Gemma (2), Mistral (1), Microsoft Phi (2)

All eight cells append to `data/explanations.jsonl`. Re-run any cell after an interruption — `inference.py` skips already-done jokes.

**Open this notebook with:**
```bash
conda activate applesVsOranges
jupyter notebook run_inference.ipynb
```

---
## Sections
1. Environment check
2. Ollama setup — point storage at D: drive + pull all 8 models
3. Smoke test — 4 jokes per model
4. Full run — R1-Distill-Llama-8B
5. Full run — Llama 3.1 8B
6. Full run — Llama 3.2 3B
7. Full run — Gemma 2 2B
8. Full run — Gemma 2 9B
9. Full run — Mistral 7B
10. Full run — Phi-3 Mini
11. Full run — Phi-3 Medium
12. Inspect combined output
13. Next steps — judge + analyze

---
## Section 1 — Environment Check

In [17]:
# Imports + path setup
import sys, os, json, time, subprocess
from pathlib import Path

sys.path.insert(0, "src")

from inference import main as inference_main
from inference_backend import get_inference_backend, OllamaBackend

print(f"Python  : {sys.version.split()[0]}")
print(f"CWD     : {Path.cwd()}")
print(f"jokes   : {'OK' if Path('data/jokes.jsonl').exists() else 'MISSING — run preprocess.py first'}")
print(f"output  : data/explanations.jsonl (will be created/appended)")
print()
print("Imports OK")

Python  : 3.10.19
CWD     : /mnt/d/ApplesVSOranges
jokes   : OK
output  : data/explanations.jsonl (will be created/appended)

Imports OK


In [18]:
# GPU check — matters for all 5 models since they all run via Ollama locally
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM    : {vram_gb:.1f} GB")
        if vram_gb < 8:
            print("WARN    : <8GB VRAM — Gemma 2B and Llama 3B will work; 8B/9B may need CPU offload")
    else:
        print("WARN    : no CUDA GPU detected — inference will run on CPU (much slower, but works)")
        print("          expect ~5-10x longer than the time estimates in each cell")
except ImportError:
    print("torch not installed — Ollama will use whatever it detects on its own")

GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.8 GB


---
## Section 2 — Ollama Setup (Models on D: Drive)

By default, Ollama stores models in the user home directory (typically on C:),
which fills up fast — this lineup needs ~21 GB total. We redirect storage to
the D: drive via the `OLLAMA_MODELS` environment variable.

### 2a. Choose your storage location

Pick the path Ollama should use for model storage. Defaults below assume
WSL on Windows; adjust if you're on a different setup.

| OS | Path to use |
|---|---|
| WSL (Windows) | `/mnt/d/ollama-models` |
| Native Windows | `D:\ollama-models` |
| Native Linux  | `/mnt/d/ollama-models` (or wherever your "D" drive mounts) |
| macOS | (no D drive — use `/Volumes/<external>/ollama-models` or skip this section) |

In [19]:
# Configure here — change if your D: mount path is different
MODELS_DIR = "/mnt/d/ollama-models"

# Ensure the directory exists
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
print(f"Target storage location: {MODELS_DIR}")
print(f"Directory exists       : {Path(MODELS_DIR).exists()}")

# Disk space check
import shutil
total, used, free = shutil.disk_usage(MODELS_DIR)
print(f"Free space on D:       : {free / 1e9:.1f} GB")
if free < 40 * 1e9:
    print("WARN    : <40 GB free — you may run out of space (8-model lineup needs ~36 GB)")
else:
    print("OK      : enough space for the full lineup")

Target storage location: /mnt/d/ollama-models
Directory exists       : True
Free space on D:       : 256.2 GB
OK      : enough space for the full lineup


### 2b. Stop the daemon, set `OLLAMA_MODELS`, restart

Ollama only honors `OLLAMA_MODELS` if it's set **before** the daemon starts.
Setting it from Python won't help if the daemon's already running.

**Run these in a WSL terminal (not in this notebook):**

```bash
# 1. Stop any running ollama daemon
pkill ollama 2>/dev/null
sleep 2

# 2. Optional — move any existing models to D: drive (so they're not orphaned)
#    If you already have qwen2.5:7b pulled for the judge, this preserves it.
if [ -d ~/.ollama/models ] && [ "$(ls -A ~/.ollama/models 2>/dev/null)" ]; then
    echo "Moving existing models to D: drive..."
    mkdir -p /mnt/d/ollama-models
    mv ~/.ollama/models/* /mnt/d/ollama-models/
fi

# 3. Set OLLAMA_MODELS for this session and start the daemon
export OLLAMA_MODELS=/mnt/d/ollama-models
ollama serve &
sleep 3

# 4. (Optional) Make it permanent — add to ~/.bashrc or ~/.zshrc:
echo 'export OLLAMA_MODELS=/mnt/d/ollama-models' >> ~/.bashrc
```

After the daemon restarts, run the next cell to verify.

In [20]:
# Verify the daemon is running and using the D: drive

import requests

OLLAMA_URL = "http://localhost:11434"

def ollama_running():
    try:
        requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        return True
    except Exception:
        return False

if not ollama_running():
    print("✗ Ollama daemon is not running.")
    print("  Run the bash block above in a WSL terminal, then re-run this cell.")
else:
    print("✓ Ollama daemon is running")

    # Check whether OLLAMA_MODELS is actually pointing where we want
    try:
        env_resp = subprocess.run(
            ["bash", "-c", "ps -ef | grep -E 'ollama serve' | grep -v grep | head -1"],
            capture_output=True, text=True, timeout=5,
        )
        proc_line = env_resp.stdout.strip()
        if proc_line:
            pid = proc_line.split()[1]
            with open(f"/proc/{pid}/environ", "rb") as f:
                env = f.read().decode("utf-8", errors="ignore").split("\x00")
            ollama_models_var = next(
                (e.split("=", 1)[1] for e in env if e.startswith("OLLAMA_MODELS=")),
                None
            )
            if ollama_models_var:
                if ollama_models_var == MODELS_DIR:
                    print(f"✓ Daemon's OLLAMA_MODELS is set to {MODELS_DIR}")
                else:
                    print(f"⚠ Daemon's OLLAMA_MODELS is {ollama_models_var}")
                    print(f"  Expected: {MODELS_DIR}")
                    print(f"  Stop the daemon and restart with the right env var.")
            else:
                print(f"⚠ Daemon does NOT have OLLAMA_MODELS set — using default location")
                print(f"  This means new pulls will go to ~/.ollama/models, not {MODELS_DIR}")
                print(f"  Stop the daemon and restart it with OLLAMA_MODELS={MODELS_DIR}")
    except Exception as e:
        print(f"  (couldn't introspect daemon env: {e}; skipping that check)")

✓ Ollama daemon is running
  (couldn't introspect daemon env: [Errno 13] Permission denied: '/proc/191/environ'; skipping that check)


### 2c. Pull all eight models

In [21]:
# (ollama_tag, slug, human label, est_minutes, paper_match)
MODELS = [
    ("deepseek-r1:8b",  "r1-distill-llama-8b", "DeepSeek-R1-Distill-Llama-8B", 60,  True),
    ("llama3.1:8b",     "llama-3.1-8b",        "Llama 3.1 8B Instruct",        45,  True),
    ("llama3.2:3b",     "llama-3.2-3b",        "Llama 3.2 3B Instruct",        20,  False),
    ("gemma2:2b",       "gemma-2-2b",          "Gemma 2 2B Instruct",          15,  False),
    ("gemma2:9b",       "gemma-2-9b",          "Gemma 2 9B Instruct",          50,  False),
    ("mistral:7b",      "mistral-7b",          "Mistral 7B Instruct v0.3",     40,  False),
    ("phi3:mini",       "phi-3-mini",          "Phi-3 Mini 3.8B Instruct",     25,  False),
    ("phi3:medium",     "phi-3-medium",        "Phi-3 Medium 14B Instruct",    65,  False),
]

if not ollama_running():
    print("✗ Daemon not running — fix Section 2b first.")
else:
    tags = requests.get(f"{OLLAMA_URL}/api/tags").json()
    pulled = [m["name"] for m in tags.get("models", [])]
    print(f"Currently pulled ({len(pulled)}): {pulled}\n")

    print(f"{'Model':<22} {'Slug':<24} {'Status'}")
    print("-" * 70)
    missing = []
    for tag, slug, label, _, _ in MODELS:
        is_pulled = any(tag.split(":")[0] in p and tag.split(":")[1] in p for p in pulled)
        status = "✓ pulled" if is_pulled else "✗ MISSING"
        print(f"{tag:<22} {slug:<24} {status}")
        if not is_pulled:
            missing.append(tag)

    if missing:
        print("\nPull missing models in a WSL terminal:")
        for tag in missing:
            print(f"    ollama pull {tag}")
        print("\nApprox sizes:")
        print("    deepseek-r1:8b ~5GB | llama3.1:8b ~5GB | llama3.2:3b ~2GB")
        print("    gemma2:2b      ~1.5GB | gemma2:9b ~6GB")
        print("    mistral:7b    ~4.4GB | phi3:mini ~2.2GB | phi3:medium ~7.9GB")
        print("    total ~36GB")
        print()
        print(f"After pulling, verify they landed on D: with:")
        print(f"    du -sh {MODELS_DIR}")
    else:
        print("\n✓ All eight models are pulled and ready")
        # Verify the models are actually on D: drive
        try:
            du = subprocess.run(["du", "-sh", MODELS_DIR],
                                capture_output=True, text=True, timeout=10)
            if du.returncode == 0:
                size = du.stdout.split()[0]
                print(f"  Storage on {MODELS_DIR}: {size}")
        except Exception:
            pass

Currently pulled (9): ['phi3:medium', 'phi3:mini', 'mistral:7b', 'qwen2.5:7b-instruct-q4_K_M', 'gemma2:9b', 'gemma2:2b', 'llama3.2:3b', 'llama3.1:8b', 'deepseek-r1:8b']

Model                  Slug                     Status
----------------------------------------------------------------------
deepseek-r1:8b         r1-distill-llama-8b      ✓ pulled
llama3.1:8b            llama-3.1-8b             ✓ pulled
llama3.2:3b            llama-3.2-3b             ✓ pulled
gemma2:2b              gemma-2-2b               ✓ pulled
gemma2:9b              gemma-2-9b               ✓ pulled
mistral:7b             mistral-7b               ✓ pulled
phi3:mini              phi-3-mini               ✓ pulled
phi3:medium            phi-3-medium             ✓ pulled

✓ All eight models are pulled and ready
  Storage on /mnt/d/ollama-models: 0


### 2d. Pull missing models from the notebook (optional)

Alternative to running `ollama pull` commands in a WSL terminal — this cell
pulls any missing models directly via Ollama's HTTP `/api/pull` endpoint.
Useful if you don't want to switch windows mid-setup. Total download for all
8 models is ~36 GB; re-running this cell after a partial pull only fetches
what's still missing. Safe to interrupt with the Stop button and re-run —
Ollama resumes partial downloads automatically.

In [22]:
# Pull any missing models directly from the notebook (no terminal needed).
# Uses Ollama's HTTP /api/pull endpoint for portability — no PATH issues
# regardless of conda env / WSL setup.

import json


def get_pulled_tags():
    tags = requests.get(f"{OLLAMA_URL}/api/tags").json()
    return [m["name"] for m in tags.get("models", [])]


def is_model_pulled(tag, pulled_tags):
    """Match `gemma2:9b` against `gemma2:9b-instruct-q4_K_M`."""
    family, ver = tag.split(":")
    return any(family in p and ver in p for p in pulled_tags)


def pull_model(tag):
    """Stream the pull and print progress lines that overwrite themselves."""
    resp = requests.post(
        f"{OLLAMA_URL}/api/pull",
        json={"model": tag, "stream": True},
        stream=True,
        timeout=None,    # pulls of multi-GB models can take >5 min
    )
    resp.raise_for_status()

    on_progress_line = False
    for raw in resp.iter_lines():
        if not raw:
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue

        if "error" in data:
            if on_progress_line:
                print()
            print(f"  ✗ {data['error']}")
            return False

        status = data.get("status", "")
        total = data.get("total")
        completed = data.get("completed", 0)

        if total and total > 0:
            # Download progress for a layer — overwrite the same line.
            pct = 100 * completed / total
            msg = f"  {status}: {pct:5.1f}% ({completed/1e6:>6.0f} / {total/1e6:>6.0f} MB)"
            print(f"\r{msg}", end="", flush=True)
            on_progress_line = True
        else:
            # Status line ("verifying digest", "writing manifest", "success").
            if on_progress_line:
                print()
                on_progress_line = False
            print(f"  {status}")
    if on_progress_line:
        print()
    return True


if not ollama_running():
    print("✗ Daemon not running — fix Section 2b first.")
else:
    pulled = get_pulled_tags()
    missing = [(tag, label) for tag, _, label, _, _ in MODELS
               if not is_model_pulled(tag, pulled)]

    if not missing:
        print("✓ All 8 models already pulled — nothing to do.")
    else:
        print(f"Pulling {len(missing)} missing model(s):")
        for tag, label in missing:
            print(f"  • {tag} ({label})")
        print()

        for i, (tag, label) in enumerate(missing, 1):
            print(f"[{i}/{len(missing)}] {tag} ({label})")
            if not pull_model(tag):
                print(f"\n✗ Aborted at {tag}. Fix the error above and re-run this cell.")
                break
            print(f"  ✓ {tag} done\n")

        # Final verification — re-fetch the tag list and confirm each model.
        pulled = get_pulled_tags()
        print("Final status:")
        all_ok = True
        for tag, _, label, _, _ in MODELS:
            present = is_model_pulled(tag, pulled)
            print(f"  {'✓' if present else '✗'} {tag:<22} ({label})")
            if not present:
                all_ok = False

        if all_ok:
            try:
                du = subprocess.run(["du", "-sh", MODELS_DIR],
                                    capture_output=True, text=True, timeout=10)
                if du.returncode == 0:
                    print(f"\nStorage on {MODELS_DIR}: {du.stdout.split()[0]}")
            except Exception:
                pass
            print("\n✓ All 8 models present — ready for Section 3 onwards.")

✓ All 8 models already pulled — nothing to do.


---
## Section 3 — Smoke Test (4 jokes per model)

Quick sanity check on each model before committing to 600-joke runs.

In [23]:
SMOKE_OUTPUT = "outputs/smoketest_inference.jsonl"
Path(SMOKE_OUTPUT).unlink(missing_ok=True)

def run_smoke(tag, slug, label):
    print(f"\n{'='*60}\nSmoke: {label} ({slug})\n{'='*60}")
    sys.argv = [
        "inference.py",
        "--backend", "ollama",
        "--model-id", tag,
        "--model-slug", slug,
        "--output", SMOKE_OUTPUT,
        "--limit", "4",
        "--no-resume",
    ]
    try:
        inference_main()
    except SystemExit:
        pass
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

for tag, slug, label, _, _ in MODELS:
    run_smoke(tag, slug, label)

# Verify by counting rows actually written per model — NOT by exit code,
# because the script can exit cleanly while still failing every API call.
print(f"\n{'='*60}\nSmoke results (rows actually written):\n{'='*60}")
written_by_slug = {}
if Path(SMOKE_OUTPUT).exists():
    rows = [json.loads(l) for l in open(SMOKE_OUTPUT)]
    for r in rows:
        written_by_slug[r["model"]] = written_by_slug.get(r["model"], 0) + 1

all_ok = True
for tag, slug, label, _, _ in MODELS:
    n = written_by_slug.get(slug, 0)
    ok = n == 4
    if not ok:
        all_ok = False
    print(f"  {'✓' if ok else '✗'}  {slug:<24} {n}/4 rows")

if not all_ok:
    print("\n⚠ Some models wrote 0 rows. Most likely cause:")
    print("  (a) the model wasn't pulled — check Section 2c output")
    print("  (b) the daemon isn't actually running — check Section 2b")
    print("  (c) see outputs/inference.errors.log for the exact errors")
else:
    print("\n✓ All five models wrote rows successfully — safe to run full sections 4-8")
    # Show one sample per model
    seen = set()
    for r in rows:
        if r["model"] not in seen:
            print(f"\n[{r['model']}] {r['joke_id']}")
            print(f"  {r['explanation'][:200]}{'...' if len(r['explanation']) > 200 else ''}")
            seen.add(r["model"])

INFO inference: backend=ollama model_id=deepseek-r1:8b slug=r1-distill-llama-8b — 4 jokes to do (temp=0, sleep=0s)



Smoke: DeepSeek-R1-Distill-Llama-8B (r1-distill-llama-8b)


infer[r1-distill-llama-8b]: 100%|██████████| 4/4 [00:22<00:00,  5.51s/it]
INFO inference: done: ok=4 errors={}
INFO inference: backend=ollama model_id=llama3.1:8b slug=llama-3.1-8b — 4 jokes to do (temp=0, sleep=0s)



Smoke: Llama 3.1 8B Instruct (llama-3.1-8b)


infer[llama-3.1-8b]: 100%|██████████| 4/4 [00:13<00:00,  3.37s/it]
INFO inference: done: ok=4 errors={}
INFO inference: backend=ollama model_id=llama3.2:3b slug=llama-3.2-3b — 4 jokes to do (temp=0, sleep=0s)


...


Smoke: Phi-3 Mini 3.8B Instruct (phi-3-mini)


infer[phi-3-mini]: 100%|██████████| 4/4 [00:11<00:00,  3.00s/it]
INFO inference: done: ok=4 errors={}
INFO inference: backend=ollama model_id=phi3:medium slug=phi-3-medium — 4 jokes to do (temp=0, sleep=0s)



Smoke: Phi-3 Medium 14B Instruct (phi-3-medium)


infer[phi-3-medium]: 100%|██████████| 4/4 [00:16<00:00,  4.06s/it]
INFO inference: done: ok=4 errors={}



Smoke results (rows actually written):
  ✓  r1-distill-llama-8b      4/4 rows
  ✓  llama-3.1-8b             4/4 rows
  ✓  llama-3.2-3b             4/4 rows
  ✓  gemma-2-2b               4/4 rows
  ✓  gemma-2-9b               4/4 rows
  ✓  mistral-7b               4/4 rows
  ✓  phi-3-mini               4/4 rows
  ✓  phi-3-medium             4/4 rows

✓ All five models wrote rows successfully — safe to run full sections 4-8

[r1-distill-llama-8b] homographic_000
  This joke relies on a pun, or wordplay.

The punchline is: "They hid from the gunman in a sauna where they could sweat it out."

The pun lies in the double meaning of "hid." "Hid" sounds very similar ...

[llama-3.1-8b] homographic_000
  This joke is a play on words. The phrase "sweat it out" has a double meaning here. In one sense, people often use this expression to describe waiting out a stressful or difficult situation, as if thei...

[llama-3.2-3b] homographic_000
  This joke is a play on words. The phrase "sweat it out" 

---
## Section 4 — Full Run: DeepSeek-R1-Distill-Llama-8B

Estimated runtime: ~60 min on a consumer GPU.
Resume safe — re-run after any interruption. — **paper-match model**, your scores compare directly to the paper's reported numbers

In [24]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "deepseek-r1:8b",
    "--model-slug", "r1-distill-llama-8b",
    "--output", "data/explanations.jsonl",
    "--max-tokens", "4096",        # double the budget
    "--temperature", "0.5",         # break the deterministic loop (ran on last 2 jokes)
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=r1-distill-llama-8b


---
## Section 5 — Full Run: Llama 3.1 8B Instruct

Estimated runtime: ~45 min on a consumer GPU.
Resume safe — re-run after any interruption. — **paper-match model**, your scores compare directly to the paper's reported numbers

In [25]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "llama3.1:8b",
    "--model-slug", "llama-3.1-8b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=llama-3.1-8b


---
## Section 6 — Full Run: Llama 3.2 3B Instruct

Estimated runtime: ~20 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, frame as testing whether findings generalize

In [26]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "llama3.2:3b",
    "--model-slug", "llama-3.2-3b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=llama-3.2-3b


---
## Section 7 — Full Run: Gemma 2 2B Instruct

Estimated runtime: ~15 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, frame as testing whether findings generalize

In [27]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "gemma2:2b",
    "--model-slug", "gemma-2-2b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=gemma-2-2b


---
## Section 8 — Full Run: Gemma 2 9B Instruct

Estimated runtime: ~50 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, frame as testing whether findings generalize

In [28]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "gemma2:9b",
    "--model-slug", "gemma-2-9b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=gemma-2-9b


---
## Section 9 — Full Run: Mistral 7B Instruct v0.3

Estimated runtime: ~40 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, adds Mistral family to the lineup.

In [29]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "mistral:7b",
    "--model-slug", "mistral-7b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=mistral-7b


---
## Section 10 — Full Run: Phi-3 Mini 3.8B Instruct

Estimated runtime: ~25 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, smaller half of a clean within-Phi H4 size pair (with Phi-3 Medium).

In [30]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "phi3:mini",
    "--model-slug", "phi-3-mini",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=phi-3-mini


---
## Section 11 — Full Run: Phi-3 Medium 14B Instruct

Estimated runtime: ~65 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, larger half of the clean within-Phi H4 size pair.

In [31]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "phi3:medium",
    "--model-slug", "phi-3-medium",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 4800 explanations already in data/explanations.jsonl (600 for this model)
INFO inference: nothing to do — all 600 jokes already explained for model=phi-3-medium


---
## Section 12 — Inspect Combined Output

In [32]:
import pandas as pd

OUTPUT = Path("data/explanations.jsonl")
EXPECTED_TOTAL = len(MODELS) * 600   # 8 × 600 = 4800

if not OUTPUT.exists() or OUTPUT.stat().st_size == 0:
    print("No explanations yet — run Sections 4-8 first.")
else:
    rows = [json.loads(l) for l in OUTPUT.open()]
    df = pd.DataFrame(rows)
    df["joke_type"] = df["joke_id"].str.rsplit("_", n=1).str[0]

    pct = 100 * len(df) / EXPECTED_TOTAL
    print(f"Total explanations : {len(df):,} / {EXPECTED_TOTAL:,}  ({pct:.1f}% complete)")
    print(f"Models             : {sorted(df['model'].unique())}")
    print(f"Joke types         : {sorted(df['joke_type'].unique())}")
    print()
    print("Coverage matrix (model × joke_type — 150 in every cell when complete):")
    print(df.groupby(["model", "joke_type"]).size().unstack(fill_value=0))
    print()

    df["chars"] = df["explanation"].str.len()
    df["words"] = df["explanation"].str.split().str.len()
    print("Explanation length stats by model:")
    print(df.groupby("model")[["chars", "words"]].agg(["mean", "min", "max"]).round(0))

Total explanations : 4,800 / 4,800  (100.0% complete)
Models             : ['gemma-2-2b', 'gemma-2-9b', 'llama-3.1-8b', 'llama-3.2-3b', 'mistral-7b', 'phi-3-medium', 'phi-3-mini', 'r1-distill-llama-8b']
Joke types         : ['heterographic', 'homographic', 'non_topical', 'topical']

Coverage matrix (model × joke_type — 150 in every cell when complete):
joke_type            heterographic  homographic  non_topical  topical
model                                                                
gemma-2-2b                     150          150          150      150
gemma-2-9b                     150          150          150      150
llama-3.1-8b                   150          150          150      150
llama-3.2-3b                   150          150          150      150
mistral-7b                     150          150          150      150
phi-3-medium                   150          150          150      150
phi-3-mini                     150          150          150      150
r1-distill-llam

In [33]:
# Eyeball one explanation per (model, joke_type) — quick quality check

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    rows = [json.loads(l) for l in OUTPUT.open()]
    df = pd.DataFrame(rows)
    df["joke_type"] = df["joke_id"].str.rsplit("_", n=1).str[0]

    for (model, jt), group in df.groupby(["model", "joke_type"]):
        sample = group.iloc[0]
        print(f"\n[{model} | {jt}]  joke_id={sample['joke_id']}")
        print(f"  {sample['explanation'][:300]}")
        if len(sample["explanation"]) > 300:
            print(f"  ... ({len(sample['explanation'])} chars total)")


[gemma-2-2b | heterographic]  joke_id=heterographic_000
  The joke plays on the common phrase "halfway up a mountain."  It sets up an expectation of a literal climb, but then introduces a twist by using "alleged" as a word that implies something is being claimed or stated falsely. 

This creates humor because it suggests Tom's claim about being halfway up 
  ... (418 chars total)

[gemma-2-2b | homographic]  joke_id=homographic_000
  The joke plays on the double meaning of "sweat it out."  

* **Literal:** In a sauna, people sweat to relax and detoxify.
* **Figurative:** To "sweat it out" means to endure hardship or difficult situations. 

The humor lies in the unexpected juxtaposition of these two meanings. The audience expects
  ... (498 chars total)

[gemma-2-2b | non_topical]  joke_id=non_topical_000
  The joke is a play on words using the double meaning of "blow" and "swallow." 

* **Whale's plan:** The male whale wants to sink the whaling ship by blowing air out of his blowhole.

---
## Section 13 — Next Steps

When `data/explanations.jsonl` reaches 3,000 rows (5 models × 600 jokes), run Member 1's pipeline against your outputs:

```bash
# Judge YOUR explanations (Qwen 7B via the same Ollama daemon)
python src/judge.py \
    --explanations data/explanations.jsonl \
    --output outputs/ratings_judge_ours.jsonl \
    --backend ollama

# Automatic metrics
python src/metrics.py \
    --explanations data/explanations.jsonl \
    --output outputs/metrics_ours.csv

# Figures + hypothesis tests
python src/analyze.py \
    --ratings outputs/ratings_judge_ours.jsonl \
    --metrics outputs/metrics_ours.csv \
    --outdir outputs/ours/
```

If `metrics.py` or `analyze.py` don't have the relevant flags yet, that's a small argparse change.

### Caveats for the writeup

1. **Two paper matches, three extension models.** Frame in two parts:
   - For `r1-distill-llama-8b` and `llama-3.1-8b`: direct comparison to the paper's reported numbers; document the gap.
   - For `llama-3.2-3b`, `gemma-2-2b`, `gemma-2-9b`: framed as *"do the paper's findings generalize to newer / different-family models?"*

2. **Two H4 tests, one cleaner than the other.**
   - **Clean H4 test (Gemma 2B vs 9B)** — same generation, same training run, only size differs. This is the strongest evidence.
   - **Confounded H4 test (Llama 3.1 8B vs 3.2 3B)** — different generations, so the comparison conflates size and generation improvements. Worth running for cross-validation but flag the confound.

3. **No judge family-bias** — all five inference models are Llama-derived or Gemma; the Qwen 7B judge has no shared-family inflation risk. Worth saying explicitly because it strengthens the evaluation methodology.

4. **All-local stack, fully reproducible** — no API calls, no closed-source dependencies. Anyone with Ollama can rerun your entire experiment with the model tags listed in the README. Defensible framing: *"we deliberately use only freely-runnable open-source models to ensure full reproducibility of the inference step."*